# Chimera — Environment Setup (`chimera_env`)

Bootstraps a **mamba** environment named `chimera_env` that is kept in sync with [`chimera/pyproject.toml`](../pyproject.toml) by installing the `chimera` package itself in **editable** mode. The install is staged in three steps so that `torch-scatter` (in the `scatter` extra) builds cleanly — see below.

**What the env contains**
- **Core** (always installed): scvi-tools, scanpy, anndata, mudata, torch, lightning, wandb, optuna, pybiomart, scib-metrics, numpy/pandas/scipy/scikit-learn, jupyter, papermill, matplotlib, seaborn.
- **`dev`** — pytest, pytest-cov, mypy, ruff.
- **`graph`** — torch-geometric (pure-Python GNN layer library).
- **`scatter`** — torch-scatter (C++ scatter ops used by GIN; built in Stage 3).
- **`attributions`** — captum + shap for feature-attribution probes on the VAE latent and GIN gates.
- **`protein`** — biopython, fair-esm (Meta's ESM-2 protein language model, already used for Chimera's GIN input features), py3Dmol for inline 3D structure viz.
- **`evolution`** — dendropy for phylogenetic-tree manipulation and ortholog-evolution analyses.
- **`data`** — gdown for cached dataset downloads.

**Platform-aware torch install**
- **macOS** (arm64 or x86_64): default PyPI → CPU + MPS wheel. **No CUDA** is installed (PyPI does not ship CUDA wheels for Darwin in the first place).
- **Linux with NVIDIA GPU** (detected via `nvidia-smi`): default PyPI → CUDA wheel.
- **Linux without a GPU**: routed to `https://download.pytorch.org/whl/cpu` so we don't pull ~2 GB of unusable CUDA artifacts.

**Install stages**
1. **torch** — installed first with the platform-appropriate wheel so later steps can `import torch`.
2. **chimera[dev,graph,attributions,protein,evolution,data]** — editable install of `chimera` plus every pure-Python extra.
3. **torch-scatter** — installed with `--no-build-isolation` so its `setup.py import torch` succeeds against the env's torch from stage 1. Without this flag, pip builds the wheel in a fresh venv where torch is absent and the build fails (`ModuleNotFoundError: No module named 'torch'`). Version pin mirrors the `scatter` extra in `pyproject.toml`.

**Behavior**
- If `chimera_env` already exists, the notebook is a no-op (skips).
- If it does not, the notebook creates the env (Python 3.12), runs the three install stages, registers a `chimera_env` Jupyter kernel, and prints the full package manifest at the end.
- Because the install is editable, any edit to the `chimera/` source tree is picked up without reinstalling. If `pyproject.toml` **dependencies** change, re-run the "Force re-sync" cell at the bottom (or `mamba env remove -n chimera_env` and re-run this notebook).

**How to run**
- Run this notebook top-to-bottom with any Python kernel that has a working `subprocess` (e.g. the base mamba `python3`). You do not need `chimera_env` to exist to run this notebook.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

ENV_NAME = "chimera_env"
PYTHON_VERSION = "3.12"


def find_chimera_root() -> Path:
    """Locate the chimera project root by searching for its pyproject.toml.

    Must be cwd-independent because the notebook may be executed by papermill
    from the workspace root, by nbconvert from chimera/notebooks/, or from any
    other directory. We resolve CHIMERA_ROOT from the first location in which
    pyproject.toml declares `name = "chimera"`.
    """
    candidates = [
        Path.cwd() / "chimera",    # papermill from workspace root
        Path.cwd().parent,         # nbconvert from chimera/notebooks/
        Path.cwd(),                # run from inside chimera/ itself
    ]
    # Walk up from cwd too, in case we were invoked from deeper nesting.
    cur = Path.cwd().resolve()
    for _ in range(6):
        candidates.append(cur)
        if cur.parent == cur:
            break
        cur = cur.parent

    for c in candidates:
        pp = (c / "pyproject.toml")
        if pp.exists():
            try:
                if 'name = "chimera"' in pp.read_text():
                    return c.resolve()
            except OSError:
                continue
    raise FileNotFoundError(
        "Could not locate chimera's pyproject.toml. Run this notebook from the "
        "chimera/ project root, from chimera/notebooks/, or from the workspace root "
        "(where `chimera/pyproject.toml` is reachable)."
    )


CHIMERA_ROOT = find_chimera_root()
PYPROJECT = CHIMERA_ROOT / "pyproject.toml"

MAMBA = shutil.which("mamba") or shutil.which("micromamba") or shutil.which("conda")
assert MAMBA is not None, "Neither mamba, micromamba, nor conda was found on PATH."

print(f"Using package manager: {MAMBA}")
print(f"Chimera project root : {CHIMERA_ROOT}")
print(f"pyproject.toml       : {PYPROJECT}")
print(f"Target environment   : {ENV_NAME} (python={PYTHON_VERSION})")

Using package manager: /Users/kemalinecik/tools/apps/mamba/bin/mamba
Chimera project root : /Users/kemalinecik/git_nosync/master_chimera/chimera
pyproject.toml       : /Users/kemalinecik/git_nosync/master_chimera/chimera/pyproject.toml
Target environment   : chimera_env (python=3.12)


In [2]:
def run(cmd, check=True):
    """Run a shell command, stream output, and raise on non-zero exit if check=True."""
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")
    return proc.returncode


def env_exists(name: str) -> bool:
    out = subprocess.run(
        [MAMBA, "env", "list"], capture_output=True, text=True, check=True
    ).stdout
    for line in out.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name_col = line.split()[0]
        if name_col == name:
            return True
    return False

In [3]:
import platform

OS = platform.system()      # 'Darwin', 'Linux', 'Windows'
MACHINE = platform.machine()  # 'arm64', 'x86_64', ...

def _linux_has_nvidia_gpu() -> bool:
    """True iff nvidia-smi is on PATH and returns 0 — our proxy for a usable CUDA GPU."""
    from shutil import which
    if which("nvidia-smi") is None:
        return False
    probe = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    return probe.returncode == 0

# Decide the torch wheel channel for THIS machine.
# macOS never has CUDA (PyPI ships no CUDA wheels for Darwin), so default PyPI gives CPU+MPS.
# Linux PyPI default is CUDA — good if a GPU is present, wasteful (~2 GB) if not; route to CPU wheel if no GPU.
# Windows: default PyPI (CUDA if present, else CPU).
if OS == "Darwin":
    TORCH_INDEX_URL = None
    PLATFORM_NOTE = f"macOS/{MACHINE}: PyPI default → CPU + MPS torch. No CUDA will be installed (not available on macOS)."
elif OS == "Linux":
    if _linux_has_nvidia_gpu():
        TORCH_INDEX_URL = None
        PLATFORM_NOTE = "Linux + NVIDIA GPU detected: PyPI default → CUDA torch wheel."
    else:
        TORCH_INDEX_URL = "https://download.pytorch.org/whl/cpu"
        PLATFORM_NOTE = "Linux, no NVIDIA GPU detected: using CPU-only torch wheel index."
else:
    TORCH_INDEX_URL = None
    PLATFORM_NOTE = f"{OS}/{MACHINE}: PyPI default."

print(PLATFORM_NOTE)
print(f"TORCH_INDEX_URL = {TORCH_INDEX_URL!r}")

macOS/arm64: PyPI default → CPU + MPS torch. No CUDA will be installed (not available on macOS).
TORCH_INDEX_URL = None


In [4]:
if env_exists(ENV_NAME):
    print(f"Environment '{ENV_NAME}' already exists — skipping creation and install.")
    print("If you changed dependencies in pyproject.toml, either:")
    print("  (a) run the 'Force re-sync' cell below, or")
    print(f"  (b) `mamba env remove -n {ENV_NAME}` and re-run this notebook.")
else:
    print(f"Creating environment '{ENV_NAME}' with python={PYTHON_VERSION}...")
    run(f"mamba create -n {ENV_NAME} python={PYTHON_VERSION} pip -y")

    # STAGE 1 — install torch BEFORE chimera, so later build-from-source deps
    # (notably torch-scatter, whose setup.py does `import torch`) can find it.
    # On macOS the PyPI default wheel is CPU+MPS; on Linux with NVIDIA it's the
    # CUDA wheel; on Linux without a GPU we route to the CPU-only index URL.
    torch_cmd = f"mamba run -n {ENV_NAME} pip install torch"
    if TORCH_INDEX_URL is not None:
        torch_cmd += f" --index-url {TORCH_INDEX_URL}"
    print(f"\n[Stage 1/3] Installing torch ({PLATFORM_NOTE})...")
    run(torch_cmd)

    # STAGE 2 — install chimera in editable mode with every pure-Python extra.
    # This enumerates all extras except `scatter`; that one has a C++ build hook
    # that imports torch and must be installed separately in Stage 3 under
    # --no-build-isolation. Everything else (dev, graph, attributions, protein,
    # evolution, data) builds cleanly under pip's default build isolation.
    pure_python_extras = "dev,graph,attributions,protein,evolution,data"
    print(f"\n[Stage 2/3] Installing chimera editable + [{pure_python_extras}] extras...")
    run(f"mamba run -n {ENV_NAME} pip install -e '{CHIMERA_ROOT}[{pure_python_extras}]'")

    # STAGE 3 — install torch-scatter with --no-build-isolation so its build hook
    # imports the env's already-installed torch. Version pin mirrors the
    # `scatter` extra in pyproject.toml.
    print("\n[Stage 3/3] Installing torch-scatter (scatter extra) with --no-build-isolation...")
    run(f"mamba run -n {ENV_NAME} pip install --no-build-isolation 'torch-scatter>=2.1.0'")

    print("\nRegistering Jupyter kernel so notebooks can select 'Python (chimera_env)'...")
    run(
        f"mamba run -n {ENV_NAME} python -m ipykernel install --user "
        f"--name {ENV_NAME} --display-name 'Python ({ENV_NAME})'"
    )

    print(f"\nEnvironment '{ENV_NAME}' is ready.")

Creating environment 'chimera_env' with python=3.12...
$ mamba create -n chimera_env python=3.12 pip -y


Transaction

  Prefix: /Users/kemalinecik/tools/apps/mamba/envs/chimera_env

  Updating specs:

   - python=3.12
   - pip


  Package              Version  Build               Channel           Size
────────────────────────────────────────────────────────────────────────────
  Install:
────────────────────────────────────────────────────────────────────────────

  + ncurses                6.5  h5e97a16_3          conda-forge     Cached
  + bzip2                1.0.8  hd037594_9          conda-forge     Cached
  + libexpat             2.7.5  hf6b4638_0          conda-forge     Cached
  + liblzma              5.8.3  h8088a28_0          conda-forge     Cached
  + libzlib              1.3.2  h8088a28_2          conda-forge     Cached
  + libffi               3.5.2  hcf2aa1b_0          conda-forge     Cached
  + readline               8.3  h46df422_0          conda-forge     Cached
  + tk                  8.6.13  h010d191_3          conda-forge     Cached
  + libsqlite           3.53.0  h1b


Looking for: ['python=3.12', 'pip']


Preparing transaction: ...working... done
Verifying transaction: ...working... 

done
Executing transaction: ...working... 

done



To activate this environment, use

     $ mamba activate chimera_env

To deactivate an active environment, use

     $ mamba deactivate


[Stage 1/3] Installing torch (macOS/arm64: PyPI default → CPU + MPS torch. No CUDA will be installed (not available on macOS).)...
$ mamba run -n chimera_env pip install torch


  Using cached torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl (80.6 MB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Using cached fsspec-2026.3.0-py3-none-any.whl (202 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.

Obtaining file:///Users/kemalinecik/git_nosync/master_chimera/chimera
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached scvi_tools-1.4.2-py3-none-any.whl.metadata (23 kB)
  Using cached anndata-0.12.11-py3-none-any.whl.metadata (9.9 kB)
  Using cached scanpy-1.12.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached mudata-0.3.6-py3-none-any.whl.metadata (10 kB)
  Using cached lightning-2.6.1-py3-none-any.whl.metadata (44 kB)
  Using cached wandb-0.26.0-py3-none-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached optuna-4.8.0-py3-non

  Using cached torch_scatter-2.1.2-cp312-cp312-macosx_11_0_arm64.whl


Registering Jupyter kernel so notebooks can select 'Python (chimera_env)'...
$ mamba run -n chimera_env python -m ipykernel install --user --name chimera_env --display-name 'Python (chimera_env)'


Installed kernelspec chimera_env in /Users/kemalinecik/Library/Jupyter/kernels/chimera_env


Environment 'chimera_env' is ready.


In [5]:
print("Verifying environment contents...\n")
run(f"mamba run -n {ENV_NAME} python -c \"import sys; print('python:', sys.version.split()[0])\"")
run(f"mamba run -n {ENV_NAME} python -c \"import chimera; print('chimera import OK from', chimera.__file__)\"")
run(f"mamba run -n {ENV_NAME} pip show chimera | head -n 6")

# Report the torch flavour that actually got installed and sanity-check against the platform.
# On macOS we expect: cuda.is_available() == False, mps.is_available() == True.
# On Linux GPU box we expect: cuda.is_available() == True.
# On Linux CPU-only we expect: cuda.is_available() == False, __version__ ending in '+cpu'.
run(
    f"mamba run -n {ENV_NAME} python -c \""
    "import torch, platform;"
    "print('torch', torch.__version__);"
    "print('  CUDA available:', torch.cuda.is_available());"
    "print('  MPS available :', getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available());"
    "print('  platform      :', platform.system(), platform.machine())"
    "\""
)

Verifying environment contents...

$ mamba run -n chimera_env python -c "import sys; print('python:', sys.version.split()[0])"


python: 3.12.13

$ mamba run -n chimera_env python -c "import chimera; print('chimera import OK from', chimera.__file__)"


chimera import OK from /Users/kemalinecik/git_nosync/master_chimera/chimera/chimera/__init__.py

$ mamba run -n chimera_env pip show chimera | head -n 6


Name: chimera
Version: 0.1.0
Summary: Cross-species scRNA-seq integration with learnable per-gene evolutionary gating
Home-page: 
Author: 
Author-email: 
$ mamba run -n chimera_env python -c "import torch, platform;print('torch', torch.__version__);print('  CUDA available:', torch.cuda.is_available());print('  MPS available :', getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available());print('  platform      :', platform.system(), platform.machine())"


torch 2.11.0
  CUDA available: False
  MPS available : True
  platform      : Darwin arm64



0

---
## Force re-sync with `pyproject.toml`

Run the cell below only when you need to refresh dependencies after editing `pyproject.toml` (e.g. adding a new pinned package). A plain `pip install -e` re-solves dependencies; no env recreation needed. For a full reset, use `mamba env remove -n chimera_env` and re-run the cells above.

In [6]:
# Uncomment to force-resync dependencies from pyproject.toml.
# Same three-stage install as the creation path above, with --upgrade on the chimera step.
# run(f"mamba run -n {ENV_NAME} pip install -e '{CHIMERA_ROOT}[dev,graph,attributions,protein,evolution,data]' --upgrade")
# run(f"mamba run -n {ENV_NAME} pip install --no-build-isolation 'torch-scatter>=2.1.0' --upgrade")

---
## Full environment manifest

Prints every package currently installed in `chimera_env` with its exact version, so the output of this notebook is a human-readable lockfile — reproducible to the byte for anyone who re-runs the notebook on the same platform. `mamba list` covers conda-installed and pip-installed packages in a single table; the editable install path for `chimera` itself is already shown by the `pip show chimera` call in the verify cell above.

In [7]:
print(f"=== Full package manifest for '{ENV_NAME}' (mamba list) ===\n")
run(f"mamba list -n {ENV_NAME}")

=== Full package manifest for 'chimera_env' (mamba list) ===

$ mamba list -n chimera_env


# packages in environment at /Users/kemalinecik/tools/apps/mamba/envs/chimera_env:
#
# Name                    Version                   Build  Channel
absl-py                   2.4.0                    pypi_0    pypi
aiohappyeyeballs          2.6.1                    pypi_0    pypi
aiohttp                   3.13.5                   pypi_0    pypi
aiosignal                 1.4.0                    pypi_0    pypi
alembic                   1.18.4                   pypi_0    pypi
anndata                   0.12.11                  pypi_0    pypi
annotated-types           0.7.0                    pypi_0    pypi
anyio                     4.13.0                   pypi_0    pypi
appnope                   0.1.4                    pypi_0    pypi
argon2-cffi               25.1.0                   pypi_0    pypi
argon2-cffi-bindings      25.1.0                   pypi_0    pypi
array-api-compat          1.14.0                   pypi_0    pypi
arrow                     1.4.0                    pypi_

0

---
## How to remove `chimera_env`

If you want to tear down the environment (e.g. to rebuild from scratch after a major dependency change, or to reclaim disk space), run the commands below in your shell. They are deliberately **not** executed by this notebook — copy-paste into a terminal, or uncomment the code cell below and re-run it.

```bash
mamba env remove -n chimera_env -y   # deletes the env
mamba clean -avvvy                   # purges downloaded packages, tarballs, indexes, etc.
```

In [8]:
# Uncomment ALL lines below to delete chimera_env and clean mamba caches.
# Echo statements narrate each step so you can follow progress in the cell output.

# run('echo "[1/2] Removing chimera_env..."')
# run(f"mamba env remove -n {ENV_NAME} -y")
# run('echo ""')
# run('echo "[2/2] Cleaning mamba package cache (this can free several GB)..."')
# run("mamba clean -avvvy")
# run('echo ""')
# run(f'echo "Done. {ENV_NAME} removed and caches cleared. Re-run the cells at the top to rebuild."')